In [1]:
import time
import numpy as np
from pynq_dpu import DpuOverlay

overlay = DpuOverlay("dpu.bit")
### Binary
# overlay.load_model("cap_binaryCNN_first.xmodel")
# obj = "binary"
### Multiclass
overlay.load_model("cap_multiCNN_first.xmodel")
obj = "multi"

In [2]:
dpu = overlay.runner

inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()

outputDim = tuple(dpu.get_output_tensors()[0].dims)
outputData = [np.empty(outputDim, dtype=np.int8)]

inputDim = tuple(dpu.get_input_tensors()[0].dims)
inputData = [np.empty(inputDim, dtype=np.int8)]
# print(inputData)
print(inputTensors)
print(inputTensors[0].dims)
print(inputTensors[0].dtype)

print(outputTensors)
print(outputTensors[0].dims)
print(outputTensors[0].dtype)

[<xir.Tensor named 'MultiClass1dCNN__MultiClass1dCNN_input_1_swim_transpose_0_fix'>]
[1, 1, 83, 1]
xint8
[<xir.Tensor named 'MultiClass1dCNN__MultiClass1dCNN_Linear_fc2__229_fix'>]
[1, 12]
xint8


In [3]:
fix_point = dpu.get_input_tensors()[0].get_attr("fix_point")
print(fix_point)

5


In [ ]:
# load data
x = np.load(f"test_{obj}_inputs.npy").astype(np.float32)
y = np.load(f"test_{obj}_labels.npy")
print(x.shape)  # should be (batch, 1, 83)

(24624, 1, 83)


In [5]:
# quant
scale = 2 ** fix_point
x_q = (x * scale).round().astype(np.int8)

In [6]:
# set input data
batchNumber = 63
inputData = [x_q[batchNumber]] # can be any of the batches 0-63 from x_q
print(y[batchNumber]) # correct label

2


### Timing Metrics

##### Latency

In [7]:
total = 0
inputData_b = [x_q[batchNumber]]
for s in range(2000):
    start = time.perf_counter()
    job = dpu.execute_async(inputData_b, outputData)
    dpu.wait(job)
    end = time.perf_counter()
    total+=(end-start)
#     print(f"Sample {b}: {(end-start)*1000:.6f} ms")
print(f"Per sample average: {(total/2000)*1000:.6f} ms")

Per sample average: 0.201684 ms


In [8]:
throughput = 2000/total
print(f"throughput = {throughput:.1f} samples per second")

throughput = 4958.2 samples per second


In [9]:
%%time
job_id = dpu.execute_async(inputData_b, outputData)
dpu.wait(job_id)

CPU times: user 1.71 ms, sys: 270 µs, total: 1.98 ms
Wall time: 1.36 ms


0

In [10]:
activations = outputData[0][0]
print(activations)

[ 23 -33  38 -12 -98 -28   8   9  -8   2 -16 -28]


In [11]:
if obj == 'binary':
    logit = activations / scale
    prob = 1 / (1 + np.exp(-logit))
    pred_class = (prob > 0.5).astype(int)
else:
    # probabilities - softmax
    logits = activations / scale   # only if your model is quantized
    exp = np.exp(logits - np.max(logits))  # stability
    probs = exp / np.sum(exp)

    pred_class = np.argmax(probs)
    confidence = probs[pred_class]
    print("Confidence:", confidence, "\n", probs)

print("Pred class:", pred_class)


Confidence: 0.26627805065726917 
 [0.16663255 0.02895639 0.26627805 0.05581491 0.00379825 0.03385346
 0.10427598 0.10758606 0.06324658 0.08644783 0.04925649 0.03385346]
Pred class: 2


---

### Performance

In [12]:
xPerf = np.load(f"testy_{obj}_inputs.npy").astype(np.float32)
yPerf = np.load(f"testy_{obj}_labels.npy")
print(xPerf.shape)  # should be (batch, 1, 83)
print(yPerf.shape)

scale = 2 ** fix_point
xPerf = (xPerf * scale).round().astype(np.int8)

(24624, 1, 83)
(24624,)


In [13]:
print("xPerf shape:", xPerf.shape)         # (B, N, 1, F)
print("yPerf shape:", yPerf.shape)         # (B, N)
print("DPU input dims:", dpu.get_input_tensors()[0].dims)
print("DPU output dims:", dpu.get_output_tensors()[0].dims)
# print("outData dtype:", outData[0].dtype)
# print("sample outData shape:", outData[0].shape)   # after one run

yPerf[0]

xPerf shape: (24624, 1, 83)
yPerf shape: (24624,)
DPU input dims: [1, 1, 83, 1]
DPU output dims: [1, 12]


np.int64(2)

In [ ]:
y_true = []
y_pred = []
out = dpu.get_output_tensors()[0].dims

inputDim = dpu.get_input_tensors()[0].dims
inputData = [np.empty(inputDim, dtype=np.int8)]

outData = [np.zeros(out, dtype=np.float32)]


for c,d in enumerate(xPerf):
    inputData = [d] # shape: (1, 1, 83) or [d]
    outData = [np.empty(out, dtype=np.int8)]
    job_id = dpu.execute_async(inputData, outData)
    dpu.wait(job_id)

    # outData[0] -> (1, 12)
    activations = outData[0] 
    # Get predictions for whole batch
    ### Binary
    # logit = activations / scale
    # prob = 1 / (1 + np.exp(-logit))
    # preds = (prob > 0.5).astype(int)
    ### Multi
    preds = np.argmax(activations, axis=1)   # (1,)

    y_pred.append(preds[0])
    y_true.append(yPerf[c])
        
    
print("done")

[1, 1, 83, 1]
[1, 12]
done


In [ ]:
print("inputData shape:", inputData[0].shape)
print("activations shape:", activations.shape)
print("y_true:", y_true[:10], "y_pred:", y_pred[:10])

inputData shape: (1, 83)
activations shape: (1, 12)
y_true: [np.int64(2), np.int64(2), np.int64(10), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(9), np.int64(2), np.int64(2)] y_pred: [np.int64(2), np.int64(2), np.int64(10), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(9), np.int64(2), np.int64(2)]


In [ ]:
num_classes = 2

if obj == 'multi':
    num_classes = 12 
    
cm = np.zeros((num_classes, num_classes), dtype=int)

for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

print(cm)

[[ 1365     6     0     4   119     8     0     0     4     0    41     3]
 [    1   105     0     0     0     0     0     0     1     0     0     0]
 [    0     0 18932     0     0     0     0     0     0     0     0     0]
 [    1     0     0   827     0     0     0     0     0     0     1     0]
 [    1     0     0     0     6     0     0     0     0     0     0     0]
 [    1     0     0     0     0     4     0     0     0     0     0     1]
 [    0     0    14     0     0     0   386     0     0     0     0     0]
 [    0     0     0     0     0     0     0   200     0     0     0     0]
 [    4    24     0     0     1     1     0     0   486     0     2     0]
 [    0     0     0     0     2     0     0     0     0   400     0     0]
 [  118     3     0     1    27     0     0     0     3     0  1465     5]
 [    5     0     0     0     0     0     0     0     0     0     0    46]]


In [ ]:
precision = np.zeros(num_classes)
recall = np.zeros(num_classes)
f1 = np.zeros(num_classes)

for i in range(num_classes):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP

    precision[i] = TP / (TP + FP + 1e-9)
    recall[i]    = TP / (TP + FN + 1e-9)
    f1[i] = 2 * (precision[i] * recall[i]) / (precision[i] + recall[i] + 1e-9)

In [ ]:
supports = cm.sum(axis=1)

tp = np.diag(cm)
fp = cm.sum(axis=0)-tp
fn = cm.sum(axis=1)-tp

In [ ]:
weighted_precision = np.sum(precision * supports) / np.sum(supports)
weighted_recall = np.sum(recall * supports) / np.sum(supports)
weighted_f1 = np.sum(f1 * supports) / np.sum(supports)

In [34]:
macro_precision = np.mean(precision)
macro_recall    = np.mean(recall)

print("Accuracy:",  np.sum(np.array(y_true,dtype=np.int64) == np.array(y_pred,dtype=np.int64)) / len(y_true))
print("Macro Precision:", macro_precision)
print("Macro Recall:", macro_recall)
print("Macro F1", np.mean(2*precision*recall / (precision + recall + 1e-10)))
print("weighted Precision:", weighted_precision)
print("weighted recall:", weighted_recall)
print("weighted f1:" ,weighted_f1)

Accuracy: 0.9836744639376218
Macro Precision: 0.8169972557544333
Macro Recall: 0.9238971722986354
Macro F1 0.8323199697017047
weighted Precision: 0.9896361644603979
weighted recall: 0.9836744639371715
weighted f1: 0.9863548780417987
